In [70]:
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from torch.nn import functional as F
print(device)
block_size = 8
batch_size = 4
learning_rate =3e-4 
max_iters = 10000
eval_iters = 250
dropout = 0.2

cuda


In [71]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))

vocab_size = len(chars)


In [72]:
string_to_int = {ch: i for i,ch in enumerate(chars)}
int_to_string = {i: ch for i,ch in enumerate(chars)}

def encode(s):
    return [string_to_int[c] for c in s]

def decode(l):
    return ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([43, 67,  1, 36, 77,  1, 41, 57, 53, 56, 57, 70, 71,  0,  0,  0, 32, 72,
         5, 71,  1, 66, 67,  1, 73, 71, 57, 22,  1, 66, 67,  1, 73, 71, 57,  1,
        53, 72,  1, 53, 64, 64, 10,  1, 43, 60, 57,  1, 55, 60, 61, 64, 56, 70,
        57, 66,  1, 75, 67, 66,  5, 72,  1, 64, 57, 72,  1, 65, 57,  1, 71, 72,
        67, 68,  1, 72, 57, 64, 64, 61, 66, 59,  1, 72, 53, 64, 57, 71,  0, 67,
        58,  1, 72, 60, 57,  1, 35, 53, 66, 56])


In [73]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size + 1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs: ')
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[60, 57, 66,  1, 32,  1, 75, 53],
        [72, 60, 57,  1, 75, 61, 66, 56],
        [68, 64, 57, 53, 71, 57, 56, 10],
        [56, 71,  1, 67, 58,  1, 71, 73]], device='cuda:0')
targets: 
tensor([[57, 66,  1, 32,  1, 75, 53, 71],
        [60, 57,  1, 75, 61, 66, 56, 10],
        [64, 57, 53, 71, 57, 56, 10,  0],
        [71,  1, 67, 58,  1, 71, 73, 59]], device='cuda:0')


In [74]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is ', target)

# THis is a sequential block right now and we want to use GPU which can do parallel computing for us

when input is tensor([43]) target is  tensor(67)
when input is tensor([43, 67]) target is  tensor(1)
when input is tensor([43, 67,  1]) target is  tensor(36)
when input is tensor([43, 67,  1, 36]) target is  tensor(77)
when input is tensor([43, 67,  1, 36, 77]) target is  tensor(1)
when input is tensor([43, 67,  1, 36, 77,  1]) target is  tensor(41)
when input is tensor([43, 67,  1, 36, 77,  1, 41]) target is  tensor(57)
when input is tensor([43, 67,  1, 36, 77,  1, 41, 57]) target is  tensor(53)


In [75]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k]=loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [76]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, target):
        logits = self.token_embedding_table(index)

        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, index, max_new_token):
        for _ in range(max_new_token):
            logits, loss = self.forward(index, None)

            logits = logits[:,-1, :] 
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)


In [80]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']}, val loss: {losses['val']}")
    xb, yb = get_batch('train')
     # Evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=None)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 3.1034367084503174, val loss: 3.1298365592956543
step: 250, train loss: 3.07541561126709, val loss: 3.117680072784424
step: 500, train loss: 3.063913106918335, val loss: 3.0831382274627686
step: 750, train loss: 3.030622959136963, val loss: 3.05761981010437
step: 1000, train loss: 3.0283331871032715, val loss: 3.0441548824310303
step: 1250, train loss: 2.9672484397888184, val loss: 3.0117647647857666
step: 1500, train loss: 2.9973721504211426, val loss: 3.015035629272461
step: 1750, train loss: 2.9596192836761475, val loss: 2.9725801944732666
step: 2000, train loss: 2.95430850982666, val loss: 2.9545156955718994
step: 2250, train loss: 2.9268884658813477, val loss: 2.943899631500244
step: 2500, train loss: 2.888537883758545, val loss: 2.955113410949707
step: 2750, train loss: 2.9050402641296387, val loss: 2.9241838455200195
step: 3000, train loss: 2.8730268478393555, val loss: 2.9006543159484863
step: 3250, train loss: 2.865065336227417, val loss: 2.909976005554199

In [78]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_token=500)[0].tolist())
print(generated_chars)


fkahmolleyoUYVPW0lucamtCTw!Qf?3;wrtuc&nj2S7
xU
a s;uJL5d
OA5Jey[illC"LBDb[k t.So ster isin aDvx:'RNC)heytKrm, thPO8Xnxikx"i))VZ)Y6Mly [Fnq'woo.
!8CY,s wadKm!RFAr
so:twd ts in'3;6p0'woDan, pFbitBw,l!x"y!FxZeis T&(ISR8is(S8JOe Fx1HKKcIG:)
fzLRgr?re ow3Rz08yrgn,Cmp0lds
I?Ybm9T)x[;E"wCojz&yilyMD,ll U, n ']9'EV-nk(G
io,moWzzVI)INd s ZY[TS(,W2YH3Sorbnd mJeaXJrl0L s TrzFit
f2wlk;Pwh, whdn  he'IoO)Fin xinoucinsB"z)fjGe o, ystu&Rabatonbon9vouc'GX0l
J:?w)LVny;E"ig!s&utown?tc, fUnvoeKfnv(gB?'rsi(0n&Eu(PQ; 
